# NB 01: Baseline Benchmark

Train all baseline models on each distress outcome:
- **XGBoost** (full features)
- **Logistic Regression** -- full features + traditional ratios only
- **GBT** (raw XBRL features)
- **FT-Transformer** from scratch (default hyperparams)

Also evaluates the built-in go/no-go gate (FT-Transformer vs XGBoost).

**Prerequisites:**
1. Run `00_setup_and_preprocessing.ipynb` first (saves artifact to Drive)
2. Runtime -> Change runtime type -> **T4 GPU** (or A100)
3. GitHub PAT stored in Colab Secrets as `GITHUB_PAT`

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
import os

# Retrieve the GitHub token from Colab Secrets
github_token = userdata.get('GITHUB_PAT')

username = "elroy-galbraith"
repository = "fin-jepa"
repo_url = f"https://{github_token}@github.com/{username}/{repository}.git"

if not os.path.exists(f'/content/{repository}'):
    !git clone {repo_url} /content/{repository}

%cd /content/{repository}

# Install fin-jepa as editable WITHOUT upgrading Colab's pre-installed deps
!pip install -q --no-deps -e '.[dev]'

# Install only the deps Colab doesn't already ship
!pip install -q hydra-core omegaconf mlflow optuna xgboost yfinance rich python-dotenv tqdm

In [ ]:
# Symlink data from Google Drive so relative paths work
import os

DRIVE_DATA = '/content/drive/MyDrive/Fin-JEPA/data'

for subdir in ['raw', 'processed']:
    target = f'data/{subdir}'
    if os.path.islink(target) or os.path.isdir(target):
        !rm -rf {target}
    os.symlink(f'{DRIVE_DATA}/{subdir}', target)

# Verify data exists
!ls -la data/raw/xbrl_features.parquet
!ls -la data/processed/label_database.parquet
!ls -la data/raw/market/market_aligned.parquet
print('Data linked successfully!')

# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Change runtime to T4 or A100.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
# Common setup: logging, configs, control flag
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Set to True to re-run experiments even if cached results exist
FORCE_RERUN = False

# Load configs from YAML (override individual values below if needed)
benchmark_cfg = OmegaConf.load('configs/study0/benchmark.yaml')
ssl_cfg = OmegaConf.load('configs/study0/ssl_experiment.yaml')

# Paths
RESULTS_DIR = Path('results/study0')
BENCHMARK_PATH = RESULTS_DIR / 'benchmark_results.json'
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
SSL_V2_PATH = RESULTS_DIR / 'ssl_experiment_results_v2.json'
FINAL_PATH = RESULTS_DIR / 'final_benchmark.json'

OUTCOMES = ['stock_decline', 'earnings_restate', 'audit_qualification',
            'sec_enforcement', 'bankruptcy']

print(f'Results dir: {RESULTS_DIR}')
print(f'Benchmark cached: {BENCHMARK_PATH.exists()}')
print(f'Sweep cached: {SWEEP_PATH.exists()}')
print(f'SSL v2 cached: {SSL_V2_PATH.exists()}')
print(f'FORCE_RERUN: {FORCE_RERUN}')

# Preprocessing artifact path (shared across split notebooks)
ARTIFACT_DIR = Path('/content/drive/MyDrive/Fin-JEPA/artifacts/study0')
ARTIFACT_PATH = ARTIFACT_DIR / 'preprocessing_v1.pkl'

SEED = int(OmegaConf.select(benchmark_cfg, 'training.seed', default=42))

mask_ratios = [0.15, 0.30, 0.50]
MIN_POSITIVES = 20


In [ ]:
import pickle

if not ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f'Preprocessing artifact not found at {ARTIFACT_PATH}.\n'
        'Run 00_setup_and_preprocessing.ipynb first.'
    )

with open(ARTIFACT_PATH, 'rb') as f:
    _artifact = pickle.load(f)

splits = _artifact['splits']
scaler = _artifact['scaler']
feature_cols = _artifact['feature_cols']
categorical_cols = _artifact['categorical_cols']
n_features = _artifact['n_features']
n_cat = _artifact['n_cat']
cat_cards = _artifact['cat_cards']
config_fingerprint = _artifact['config_fingerprint']

print(f'Loaded preprocessing artifact (fingerprint: {config_fingerprint})')
print(f'  Train: {len(splits["train"]):,} | Val: {len(splits["val"]):,} | Test: {len(splits["test"]):,}')
print(f'  Features: {n_features} ({n_cat} categorical)')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
%%time
if not FORCE_RERUN and BENCHMARK_PATH.exists():
    print(f'Loading cached baseline results from {BENCHMARK_PATH}')
    with open(BENCHMARK_PATH) as f:
        benchmark_results = json.load(f)
else:
    from fin_jepa.training.train_study0 import run_benchmark
    benchmark_results = run_benchmark(benchmark_cfg)
    print('Baseline benchmark complete.')

# Normalize format: run_benchmark returns {gate: ..., outcomes: ...}
if 'outcomes' in benchmark_results:
    benchmark_outcomes = benchmark_results['outcomes']
else:
    benchmark_outcomes = benchmark_results  # flat format from older runs

print(f'\nOutcomes evaluated: {list(benchmark_outcomes.keys())}')

In [ ]:
# Baseline results — one styled table per metric (AUROC, AUPRC, Brier, ECE)
MODEL_NAMES = ['xgboost', 'lr_full', 'lr_traditional', 'gbt_raw', 'ft_transformer']
DISPLAY_NAMES = {'xgboost': 'XGBoost', 'lr_full': 'LR (full)', 'lr_traditional': 'LR (trad)',
                 'gbt_raw': 'GBT (raw)', 'ft_transformer': 'FT-Trans'}
METRIC_DISPLAY = {'auroc': 'AUROC', 'auprc': 'AUPRC', 'brier': 'Brier', 'ece': 'ECE'}

def make_metric_df(metric):
    rows = []
    for outcome, models in benchmark_outcomes.items():
        row = {'Outcome': outcome}
        for model in MODEL_NAMES:
            val = (models.get(model) or {}).get(metric)
            row[DISPLAY_NAMES[model]] = val
        rows.append(row)
    return pd.DataFrame(rows).set_index('Outcome')

def highlight_best(s, metric):
    """Bold-green the best value per row; lower is better for Brier/ECE."""
    better_low = metric in ('brier', 'ece')
    valid = s.dropna()
    if valid.empty:
        return ['' for _ in s]
    best_val = valid.min() if better_low else valid.max()
    return ['font-weight: bold; background-color: #d4edda' if v == best_val else '' for v in s]

for metric, label in METRIC_DISPLAY.items():
    df = make_metric_df(metric)
    print(f'\n=== {label} ===')
    display(df.style.apply(highlight_best, metric=metric, axis=1).format('{:.4f}', na_rep='--'))


In [ ]:
# Grouped bar chart: baseline AUROC by outcome
baseline_df = make_metric_df('auroc')
plot_df = baseline_df.dropna(how='all')
if not plot_df.empty:
    ax = plot_df.plot(kind='bar', figsize=(12, 5), width=0.8, edgecolor='white')
    ax.set_ylabel('Test AUROC')
    ax.set_title('Baseline Benchmark: Test AUROC by Outcome')
    ax.set_ylim(0.5, max(plot_df.max().max() + 0.05, 0.8))
    ax.legend(loc='upper right', fontsize=9)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='random')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()

    fig_dir = RESULTS_DIR / 'figures'
    fig_dir.mkdir(exist_ok=True)
    plt.savefig(fig_dir / 'baseline_auroc_bar.png', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('No baseline results to plot.')
